# 文本分类实例

## Step1 导入相关包

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## Step2 加载数据

In [2]:
import pandas as pd

# 数据集是 https://github.com/SophonPlus/ChineseNlpCorpus
data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [3]:
# 去掉空行
data = data.dropna()
data

,label,review
0,1,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."
1,1,商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!
2,1,早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。
3,1,宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小...
4,1,"CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风"
...,...,...
7761,0,尼斯酒店的几大特点：噪音大、环境差、配置低、服务效率低。如：1、隔壁歌厅的声音闹至午夜3点许...
7762,0,盐城来了很多次，第一次住盐阜宾馆，我的确很失望整个墙壁黑咕隆咚的，好像被烟熏过一样家具非常的...
7763,0,看照片觉得还挺不错的，又是4星级的，但入住以后除了后悔没有别的，房间挺大但空空的，早餐是有但...
7764,0,我们去盐城的时候那里的最低气温只有4度，晚上冷得要死，居然还不开空调，投诉到酒店客房部，得到...


In [4]:
data.iloc[0]

,0
label,1
review,"距离川沙公路较近,但是公交指示不对,如果是""蔡陆线""的话,会非常麻烦.建议用别的路线.房间较..."


In [5]:
data.iloc[0]["review"]

'距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.'

In [6]:
data.iloc[0]["label"]

np.int64(1)

## Step3 创建 Dataset

In [7]:
from torch.utils.data import Dataset


class MyDataset(Dataset):

    def __init__(self) -> None:
        super().__init__()
        self.data = pd.read_csv("./ChnSentiCorp_htl_all.csv")
        self.data = self.data.dropna()

    def __getitem__(self, index):
        return self.data.iloc[index]["review"], self.data.iloc[index]["label"]

    def __len__(self):
        return len(self.data)

In [8]:
dataset = MyDataset()
for i in range(5):
    print(dataset[i])

('距离川沙公路较近,但是公交指示不对,如果是"蔡陆线"的话,会非常麻烦.建议用别的路线.房间较为简单.', np.int64(1))
('商务大床房，房间很大，床有2M宽，整体感觉经济实惠不错!', np.int64(1))
('早餐太差，无论去多少人，那边也不加食品的。酒店应该重视一下这个问题了。房间本身很好。', np.int64(1))
('宾馆在小街道上，不大好找，但还好北京热心同胞很多~宾馆设施跟介绍的差不多，房间很小，确实挺小，但加上低价位因素，还是无超所值的；环境不错，就在小胡同内，安静整洁，暖气好足-_-||。。。呵还有一大优势就是从宾馆出发，步行不到十分钟就可以到梅兰芳故居等等，京味小胡同，北海距离好近呢。总之，不错。推荐给节约消费的自助游朋友~比较划算，附近特色小吃很多~', np.int64(1))
('CBD中心,周围没什么店铺,说5星有点勉强.不知道为什么卫生间没有电吹风', np.int64(1))


## Step4 划分数据集

In [9]:
from torch.utils.data import random_split


trainset, validset = random_split(dataset, lengths=[0.9, 0.1])
len(trainset), len(validset)

(6989, 776)

In [10]:
for i in range(10):
    print(trainset[i])
    print(trainset[i][0])
    print(trainset[i][1])
    print()

('不知道什么原因，这个酒店目前的价格做了下调，而且周末居然有促销价。不错！这次入住的是大床房，依旧是紧促的布局，舒服的床，总体来说，还是很不错的！', np.int64(1))
不知道什么原因，这个酒店目前的价格做了下调，而且周末居然有促销价。不错！这次入住的是大床房，依旧是紧促的布局，舒服的床，总体来说，还是很不错的！
1

('首先酒店所在位置很隐蔽，看样子是网上订单的客人比较多，价格便宜，相对的房间硬件设施和服务就差些。我房子的空调竟然是个二手的。上面还贴着大头照。前台对你也爱理不理的。如果囊中羞涩可以考虑该酒店，否则还是去别处吧。第二天换的那个不远的天财酒店还不错。价格也合理328元', np.int64(0))
首先酒店所在位置很隐蔽，看样子是网上订单的客人比较多，价格便宜，相对的房间硬件设施和服务就差些。我房子的空调竟然是个二手的。上面还贴着大头照。前台对你也爱理不理的。如果囊中羞涩可以考虑该酒店，否则还是去别处吧。第二天换的那个不远的天财酒店还不错。价格也合理328元
0

('房间有法式风格，尤其浴室和化妆间的布置。不过淋浴房的防水做的不好，洗完澡浴室地面全是水，差点摔跤。宾馆反馈2008年4月11日：非常感谢您的光临和对我们饭店的关注和肯定。对于您提出的不足我们将努力改进，力求把工作做得更好，给宾客提供更优质、更舒适的硬件设施和软件服务。', np.int64(1))
房间有法式风格，尤其浴室和化妆间的布置。不过淋浴房的防水做的不好，洗完澡浴室地面全是水，差点摔跤。宾馆反馈2008年4月11日：非常感谢您的光临和对我们饭店的关注和肯定。对于您提出的不足我们将努力改进，力求把工作做得更好，给宾客提供更优质、更舒适的硬件设施和软件服务。
1

('位置还行,而且旁边就有个1+1超市.唯一让我不爽的是,入住后不到一小时,因行程有变,要退房,竟然称收半天房租!!!而我入住后就去吃饭,怎么样都不肯让步,真郁闷!!!看来还是住石歧好了.', np.int64(0))
位置还行,而且旁边就有个1+1超市.唯一让我不爽的是,入住后不到一小时,因行程有变,要退房,竟然称收半天房租!!!而我入住后就去吃饭,怎么样都不肯让步,真郁闷!!!看来还是住石歧好了.
0

('之前就听说苏州万丽是苏州生意最好，房价最高，也是业内人士最推崇的酒店，远胜于喜来登，香格里拉，索菲特

In [11]:
for i in range(10):
    print(validset[i])
    print(validset[i][0])
    print(validset[i][1])
    print()

('周边环境不太好，比较乱。房间设施简陋，收费偏贵。服务不太到位，中午1点你正在休息时，服务员来打扰要清理房间,真是令人烦恼。结果让他们下午晚些来，结果这一天房间也没打扫。这样的服务在别处是不可能遇到的。', np.int64(0))
周边环境不太好，比较乱。房间设施简陋，收费偏贵。服务不太到位，中午1点你正在休息时，服务员来打扰要清理房间,真是令人烦恼。结果让他们下午晚些来，结果这一天房间也没打扫。这样的服务在别处是不可能遇到的。
0

('很破的宾馆，号称四星不过实际的情况比好一点的2星都差，早饭也是惨不忍睹。', np.int64(0))
很破的宾馆，号称四星不过实际的情况比好一点的2星都差，早饭也是惨不忍睹。
0

('不错的酒店。就是早餐水果太单调。宽带直接连电信网的，速度很快。', np.int64(1))
不错的酒店。就是早餐水果太单调。宽带直接连电信网的，速度很快。
1

('酒店介绍和实际不符，达不到3星级标准，房间不紧张，还要求信用卡担保。到达现场后，客人不满意，最后选择宁可酒店扣除了房费，也选择了喀什的其他的酒店，那就是携程也未推荐的中西亚国际大酒店，价格和其相仿，但是其为四星标准。而且，携程与酒店的沟通做得不好，明明是酒店的介绍与实际有较大的偏差，而且价格比起当地旅行社的价格要高出30%，可是酒店方面还是扣除了我们订房的全部房费，我同样要考虑是否还继续使用携程作为我的选择。', np.int64(0))
酒店介绍和实际不符，达不到3星级标准，房间不紧张，还要求信用卡担保。到达现场后，客人不满意，最后选择宁可酒店扣除了房费，也选择了喀什的其他的酒店，那就是携程也未推荐的中西亚国际大酒店，价格和其相仿，但是其为四星标准。而且，携程与酒店的沟通做得不好，明明是酒店的介绍与实际有较大的偏差，而且价格比起当地旅行社的价格要高出30%，可是酒店方面还是扣除了我们订房的全部房费，我同样要考虑是否还继续使用携程作为我的选择。
0

('本次的预定是帮我同事的,据他们在酒店的感受.....总的来说很不满意!第一,该酒店说是2006年开业,实际上是2006年从别人手中接转过来的,不知道具体是哪一年的酒店,酒店设施很旧;第二,入住的当天,因预定同前台没有沟通,差点导致客人没有房间(结果通过携程的协助才得到解决:通过携程预定成功,但酒店没有留房),后来好象是抽调出的

## Step5 创建 Dataloader

In [12]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


def collate_func(batch):
    # print("batch:", batch)

    texts, labels = [], []
    for item in batch:
        texts.append(item[0])
        labels.append(item[1])
    # print("texts:", texts)
    # print("labels:", labels)

    inputs = tokenizer(
        texts,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    inputs["labels"] = torch.tensor(labels)

    return inputs


"""
batch: [
    ('房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！', np.int64(1)),
    ('酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。', np.int64(1)),
    ('入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。', np.int64(1))
]

texts: [
    '房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！',
    '酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。',
    '入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。'
]

labels: [np.int64(1), np.int64(1), np.int64(1)]
"""

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


"\nbatch: [\n    ('房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！', np.int64(1)),\n    ('酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。', np.int64(1)),\n    ('入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚度不够5星的。家具大都是原色的，地毯也是颜色的，不亮丽。电视陈旧，遥控器不灵了。比较好玩的是房间里的电水壶的大个头，好像还是进口的。还有早餐分中西常规、粤式(我倾向于这个)和日式的3个厅，不过只能选择其中一个，到10点结束。走廊是分片的感应灯。有夜床服务、午茶小饼干1块、枕头上会放一张别着小鲜花的印有诗句的卡纸，赞！洁具质量不好，除了肥皂。无烟楼层有客人在走廊和房间(房门洞开)抽烟，无人制止。缺少报纸供应。盥洗室放有女孩子可应急的发筋，细心，再赞一个。套房的景色很不错，离海倒是确实近，就在背后。稍微偏离一点商业区，不过不远，打车到五四广场3公里不到。补充点评2008年3月6日：差点忘了，我打车的司机径直把握开到东楼，拉门的先生还给我一张记录有车牌号的纸片，赞一下。另外，青岛的酒店服务普遍不错。不过饭店的服务员可能没听说过“菊花普洱”。', np.int64(1))\n]\n\ntexts: [\n    '房间很干净，设施很新，应该是义乌最好的酒店。宾馆反馈2008年4月14日：给您创造一个洁净的居住环境是我们的职责，愿酒店中的每一次都给您留下美好的记忆，谢谢您！',\n    '酒店的设施是很不错的，房间大，设施新，床和被子都是很新的。洗手间也是干、湿分离。服务员态度也是很好，出入都会打招呼。就是餐厅大堂中，适合2－3人吃饭的桌子太少了。还有就是房价在衢州应该属于偏高了。',\n    '入住西楼海景套房（据说离海更近一点），应该说总体很不错，有老五星的风范。服务也比较热情。可能是比较老或是空调或是近海的原因，房间内稍有一点异味。地毯厚

In [13]:
# 调试
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=3, shuffle=False, collate_fn=collate_func)

In [14]:
next(enumerate(trainloader))

(0,
 {'input_ids': tensor([[ 101,  679, 4761, 6887,  784,  720, 1333, 1728, 8024, 6821,  702, 6983,
          2421, 4680, 1184, 4638,  817, 3419,  976,  749,  678, 6444, 8024, 5445,
           684, 1453, 3314, 2233, 4197, 3300,  914, 7218,  817,  511,  679, 7231,
          8013, 6821, 3613, 1057,  857, 4638, 3221, 1920, 2414, 2791, 8024,  898,
          3191, 3221, 5165,  914, 4638, 2357, 2229, 8024, 5653, 3302, 4638, 2414,
          8024, 2600,  860, 3341, 6432, 8024, 6820, 3221, 2523,  679, 7231, 4638,
          8013,  102,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0],
         [ 101, 7674, 1044, 6983, 2421, 2792, 1762,  855, 5390, 2523, 7391, 5929,
          8024, 4692,

In [15]:
len(trainloader)

2330

In [16]:
from torch.utils.data import DataLoader

trainloader = DataLoader(trainset, batch_size=32, shuffle=True, collate_fn=collate_func)
validloader = DataLoader(
    validset, batch_size=64, shuffle=False, collate_fn=collate_func
)

## Step6 创建模型及优化器

In [17]:
from torch.optim import Adam

model = AutoModelForSequenceClassification.from_pretrained("hfl/rbt3")

if torch.cuda.is_available():
    model = model.cuda()

model.device

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


device(type='cuda', index=0)

In [18]:
optimizer = Adam(model.parameters(), lr=2e-5)

## Step7 训练与验证

In [19]:
def evaluate():
    model.eval()
    acc_num = 0
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            print(f"evaluate pred: {pred.long()}, labels: {batch["labels"].long()}")
            acc_num += (pred.long() == batch["labels"].long()).float().sum()
    return acc_num / len(validset)


def train(epoch=3, log_step=10):
    global_step = 0

    for ep in range(epoch):
        model.train()
        # 每个 epoch 训练全部数据
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}

            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()

            if global_step % log_step == 0:
                print(
                    f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}"
                )

            global_step += 1

        acc = evaluate()
        print(f"ep: {ep}, acc: {acc}")

## Step8 模型训练

In [20]:
train()

ep: 0, global_step: 0, loss: 0.6792185306549072
ep: 0, global_step: 10, loss: 0.6717472672462463
ep: 0, global_step: 20, loss: 0.5255171060562134
ep: 0, global_step: 30, loss: 0.6477005481719971
ep: 0, global_step: 40, loss: 0.4597216546535492
ep: 0, global_step: 50, loss: 0.23783189058303833
ep: 0, global_step: 60, loss: 0.34171104431152344
ep: 0, global_step: 70, loss: 0.40657153725624084
ep: 0, global_step: 80, loss: 0.2790922522544861
ep: 0, global_step: 90, loss: 0.4268070459365845
ep: 0, global_step: 100, loss: 0.20245224237442017
ep: 0, global_step: 110, loss: 0.2992611229419708
ep: 0, global_step: 120, loss: 0.32032665610313416
ep: 0, global_step: 130, loss: 0.23053929209709167
ep: 0, global_step: 140, loss: 0.33635789155960083
ep: 0, global_step: 150, loss: 0.25556808710098267
ep: 0, global_step: 160, loss: 0.26895344257354736
ep: 0, global_step: 170, loss: 0.2611411213874817
ep: 0, global_step: 180, loss: 0.3623621463775635
ep: 0, global_step: 190, loss: 0.21829186379909515
e

## Step9 模型预测

In [21]:
sen = "我觉得这家酒店不错，饭很好吃！"
id2_label = {0: "差评！", 1: "好评！"}
model.eval()
with torch.inference_mode():
    inputs = tokenizer(sen, return_tensors="pt")
    inputs = {k: v.cuda() for k, v in inputs.items()}
    output = model(**inputs)
    print(f"output: {output}")
    logits = output.logits
    print(f"logits: {logits}")
    pred = torch.argmax(logits, dim=-1)
    print(f"pred: {pred}")
    print(f"输入: {sen}\n模型预测结果:{id2_label.get(pred.item())}")

output: SequenceClassifierOutput(loss=None, logits=tensor([[-3.1057,  2.8289]], device='cuda:0'), hidden_states=None, attentions=None)
logits: tensor([[-3.1057,  2.8289]], device='cuda:0')
pred: tensor([1], device='cuda:0')
输入: 我觉得这家酒店不错，饭很好吃！
模型预测结果:好评！


In [22]:
from transformers import pipeline

model.config.id2label = id2_label
pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

Device set to use cuda:0


In [23]:
pipe(sen)

[{'label': '好评！', 'score': 0.9973605275154114}]